**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**04 | Governed IMU–FE Dataset Construction and Participant-Independent Splits**

![Governed source pairing and participant-disjoint data workflow](../../images/04_consolidated_workflow.svg)

**Research question:** Can the original 40-participant Essex dataset be transformed into a reproducible source-identified IMU–FE regression input dataset, with exact original time-coordinate pairing, justified exclusions and a fixed participant-disjoint evaluation protocol?

**Published target semantics:** The original [Scientific Data article](https://doi.org/10.1038/s41597-026-07429-3), Table 3, describes first and third principal femoral-neck strains `e1` and `e3` as **unitless** FE-model outputs. This is publisher-reported source metadata, not independently calibrated clinical measurement.

**Source integrity:** Preserve the original 1,897 trial identities, including 168 FE-empty records. Use checksum-verified MATLAB inputs and source-verified temporal alignment to obtain 1,729 eligible pairs; do not invent or interpolate FE targets.

**Participant separation:** Use the fixed, subject-disjoint 29/4/4 Train/Validation/Test partition of 37 eligible participants. Estimate preprocessing statistics from Training participants only; keep Test data outside model selection.


**Execution:** The simplified cells have not been rerun on the full dataset. Numerical findings described here refer to the previously completed run. Each notebook reads the current upstream files; re-execution replaces its own output files. No historical reports, hash comparisons or PASSED gates are required.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

Repository up to date.
/content/human-movement-intelligence


**01. Imports and Research Environment**

Use shared original-source and reusable paired-data modules with separate paths for public evidence and ignored numerical arrays.


In [ ]:
# =================== 01. Imports and Research Environment ====================

from python_modules.core.colab_setup import setup_colab

root = setup_colab("biomechanical_load_estimation")
from importlib.metadata import version
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans",
    title_size=12.5,
    title_weight="normal",
    label_size=11.5,
    label_weight="normal",
    tick_size=11,
    legend_size=10,
    figure_dpi=100,
    figure_size=(9, 4),
)
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ensure_matlab_archive,
    inspect_matlab_archive,
    extract_matlab_files,
)
from essex_paired_dataset import (
    build_paired_arrays,
    fixed_subject_splits,
    training_channel_stats,
    save_paired_data,
)
from essex_timebase_audit import EXPECTED_CHANNELS

study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
processed_dir = root / "data" / "processed" / "essex_femoral_neck_loading"
output_dir = study_dir / "output"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Public research outputs:", output_dir)
print("Git-ignored numerical data cache:", processed_dir)

**02. Pinned Original MATLAB Table Decoder**

Use the source-tested numerical MATLAB table reader; record the exact software version.


In [ ]:
# ================= 02. Pinned Original MATLAB Table Decoder ==================

%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat

print("Verified source reader:", version("mat-io"))

**03. Trial Registry and Time Mapping**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 03. Trial Registry and Time Mapping

registry = pd.read_csv(
    output_dir / "01_original_record_registry.csv",
    dtype={
        "local_record_key": str,
        "source_file": str,
        "subject_id": str,
        "task": str,
        "updated_trial_id": str,
        "fe_representation": str,
    },
)
mapping = pd.read_csv(
    output_dir / "03_original_time_coordinate_mapping.csv",
    dtype={"local_record_key": str, "mapping_status": str, "imu_position_indices": str},
)
archive = ensure_matlab_archive(raw_dir)
mat_files = extract_matlab_files(archive, mat_dir, inspect_matlab_archive(archive))
print("Trials:", len(registry), "| MAT files:", len(mat_files))

**04. Build Source-Exact Numerical IMU–FE Pairs**

Pair all available target signals with the original observed time indices, preserve source identity, and account for every original FE-empty record.


In [ ]:
# =============== 04. Build Source-Exact Numerical IMU–FE Pairs ===============

arrays, eligible, excluded = build_paired_arrays(
    load_from_mat, mat_dir, registry, mapping
)
print("Original numeric IMU–FE pairs:", len(eligible))
print("Original source-empty FE exclusions:", len(excluded))
eligible.to_csv(output_dir / "04_eligible_trial_registry.csv", index=False)
excluded.to_csv(output_dir / "04_excluded_trial_registry.csv", index=False)
display(eligible.head())
display(excluded.head())

**05. Source Semantics and Participant Eligibility**

Document source-published strain semantics, 42 unchanged IMU channels and the three participants with no eligible FE target.


In [ ]:
# ============= 05. Source Semantics and Participant Eligibility ==============

participant_table = (
    registry.groupby("subject_id").size().rename("original_trials").reset_index()
)
participant_table = (
    participant_table.join(
        eligible.groupby("subject_id").size().rename("eligible_pairs"), on="subject_id"
    )
    .join(excluded.groupby("subject_id").size().rename("empty_fe"), on="subject_id")
    .fillna(0)
)
participant_table[["eligible_pairs", "empty_fe"]] = participant_table[
    ["eligible_pairs", "empty_fe"]
].astype(int)
participant_table.to_csv(output_dir / "04_participant_eligibility.csv", index=False)
display(participant_table)
channel_table = pd.DataFrame(
    {
        "column_index": range(42),
        "source_column": EXPECTED_CHANNELS,
        "unit_conversion": "NONE",
    }
)
target_table = pd.DataFrame(
    {
        "source_column": ["e1", "e3"],
        "published_semantics": [
            "First principal femoral neck strain",
            "Third principal femoral neck strain",
        ],
        "published_unit": ["unitless", "unitless"],
        "independent_physical_calibration": [False, False],
    }
)
channel_table.to_csv(output_dir / "04_channel_schema.csv", index=False)
target_table.to_csv(output_dir / "04_target_schema.csv", index=False)
display(target_table)

**06. Deterministic Leakage-Free Participant Splitting**

Freeze participant-only training/validation/test identities before any model fit. Save all trial membership and activity coverage, including three excluded participants.


In [ ]:
# =========== 06. Deterministic Leakage-Free Participant Splitting ============

subjects, assignments, split_summary, activity = fixed_subject_splits(
    eligible, registry
)
display(split_summary)
subjects.to_csv(output_dir / "04_subject_split_assignments.csv", index=False)
assignments.to_csv(output_dir / "04_trial_split_assignments.csv", index=False)
split_summary.to_csv(output_dir / "04_split_trial_summary.csv", index=False)
activity.to_csv(output_dir / "04_split_activity_coverage.csv", index=False)
display(activity)

**07. Training-Only IMU Statistics**

Compute original channel means and standard deviations solely on training subjects; original numeric arrays remain untouched.


In [ ]:
# ===================== 07. Training-Only IMU Statistics ======================

train_stats = training_channel_stats(arrays, assignments)
train_stats.to_csv(output_dir / "04_training_channel_statistics.csv", index=False)
display(train_stats.head(10))
print("No time interpolation, imputation or early standardisation.")

**08. Save Current Paired Arrays**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 08. Save Current Paired Arrays

cache_path = processed_dir / "essex_imu_fe_pairs_v13.npz"
save_paired_data(cache_path, arrays, assignments)
print("Saved current arrays:", cache_path)
display(
    pd.DataFrame(
        [{"array": name, "shape": str(value.shape)} for name, value in arrays.items()]
    )
)

**09. Source Eligibility and Split Figures**

Show measured task coverage and partition counts without using any model scores or unverified FE unit transformations.


In [ ]:
# ================= 09. Source Eligibility and Split Figures ==================

coverage = (
    pd.concat(
        [
            eligible["task"].value_counts().rename("numeric_fe_pairs"),
            excluded["task"].value_counts().rename("source_empty_fe"),
        ],
        axis=1,
    )
    .fillna(0)
    .sort_index()
)
fig, ax = plt.subplots(figsize=(10, 4.5))
coverage.rename(
    columns={
        "numeric_fe_pairs": "Numeric FE pairs",
        "source_empty_fe": "Source-empty FE",
    }
).plot.bar(ax=ax, width=0.8)
ax.set_title("Original source FE eligibility by activity")
ax.set_ylabel("Original trial count")
ax.set_xlabel("Task")
ax.tick_params(axis="x", rotation=25)
save_figure(fig, output_dir / "04_eligibility_activity_coverage.png")
plt.show()
split_matrix = activity.pivot(index="task", columns="split", values="trials").fillna(0)
fig, ax = plt.subplots(figsize=(10, 4.5))
split_matrix.plot.bar(ax=ax)
ax.set_title("Eligible source trials per participant-independent split")
ax.set_ylabel("Movement trials")
ax.set_xlabel("Task")
ax.tick_params(axis="x", rotation=25)
save_figure(fig, output_dir / "04_participant_split_activity_coverage.png")
plt.show()

**10. Results Summary and Output Download**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 10. Results Summary and Output Download

display(split_summary)
print("Eligible trials:", len(eligible), "| excluded trials:", len(excluded))
print(
    "The next notebooks read 04_trial_split_assignments.csv and the current paired arrays."
)
download_outputs_from_colab(folder="biomechanical_load_estimation/output", stage="04")

**11. Scientific Interpretation and Next Research Boundary**

Notebook 04 constructs 1,729 source-identified pairs from the original 40-participant Essex dataset: 101×42 IMU trials, their 11×42 source-aligned samples, and 11×2 FE strain targets. It excludes 168 trials with source-empty FE and documents the three participants without eligible targets. No FE imputation, time interpolation or independent physical calibration is performed; the published strain targets are unitless.

**Next stage:** establish transparent regression baselines and evaluate candidate models using Training/Validation only. Preserve held-out Test participants for a separately approved final assessment after model and protocol selection.
